# Requirement 16: Independent orphan cleanup

> Enforce a short maximum lifetime and remove abandoned environments using resource metadata only, without requiring session secrets or contents.

**Covered by:** Both

**How it is fulfilled:** Every sandbox carries labels and a creation time. A cleanup job removes old sandboxes by label and age only. The platform auto-delete policy is a backstop if the job fails.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
orphans = [new_session(labels={"kind": "orphan-demo"}, egress_deny=False)[0] for _ in range(2)]
group = orchestrator()
found = list(group.list_sandboxes(labels={"kind": "orphan-demo"}))
print("abandoned sandboxes found by label:", len(found))
check("both orphans are discoverable by label", len(found) == 2)

abandoned sandboxes found by label: 2
PASS  both orphans are discoverable by label


In [3]:
# Metadata only: labels and creation time, with no secrets or contents
from datetime import datetime, timedelta, timezone
cutoff = datetime.now(timezone.utc) - timedelta(seconds=1)   # production: a few minutes beyond the maximum session time
for s in found:
    if datetime.fromisoformat(s.created_at) < cutoff:
        group.delete_sandbox(s.id)
left = list(group.list_sandboxes(labels={"kind": "orphan-demo"}))
check("sweep removed them", len(left) == 0)

PASS  sweep removed them


In [4]:
# Backstop: the platform deletes anything left after the auto-delete interval
s, _ = new_session()
policy = s.get().lifecycle.auto_delete
print("auto-delete:", policy)
check("auto-delete is enabled as a backstop", bool(policy and policy.enabled))
cleanup(s)

auto-delete: AutoDeletePolicy(enabled=True, delete_interval_seconds=900)
PASS  auto-delete is enabled as a backstop


**Caveat:** The cleanup job is custom code the orchestrator team runs. Platform auto-delete uses a time interval, not a label rule.